In [5]:
import re

# Token specification
TOKEN_SPEC = [
    ('INT',   r'\d+'),
    ('ID',    r'[A-Za-z_]\w*'),
    ('PLUS',  r'\+'),
    ('MINUS', r'-'),
    ('MUL',   r'\*'),
    ('DIV',   r'/'),
    ('LPAREN', r'\('),
    ('RPAREN', r'\)'),
    ('SKIP',  r'[ \t]+'),
    ('MISMATCH', r'.'),
]

class Token:
    def __init__(self, type_, value, position):
        self.type = type_
        self.value = value
        self.position = position
    def __repr__(self):
        if self.value:
            return f"{self.type}({self.value})"
        return self.type

def tokenize(text):
    tokens = []
    pos = 0
    pattern = '|'.join(f'(?P<{name}>{regex})' for name, regex in TOKEN_SPEC)
    for m in re.finditer(pattern, text):
        kind = m.lastgroup
        value = m.group()
        if kind == 'SKIP':
            continue
        elif kind == 'MISMATCH':
            raise SyntaxError(f"Invalid character '{value}' at position {pos}")
        tokens.append(Token(kind, value, pos))
        pos = m.end()
    tokens.append(Token('EOF', None, pos))
    return tokens


In [6]:
print(tokenize("a12 + 34*(b - 5)"))


[ID(a12), PLUS(+), INT(34), MUL(*), LPAREN((), ID(b), MINUS(-), INT(5), RPAREN()), EOF]


In [7]:
class Parser:
    def __init__(self, tokens):
        self.tokens = tokens
        self.pos = 0
        self.current_token = self.tokens[self.pos]

    def eat(self, token_type):
        if self.current_token.type == token_type:
            self.pos += 1
            self.current_token = self.tokens[self.pos]
        else:
            raise SyntaxError(
                f"Expected {token_type} at position {self.current_token.position}, "
                f"found {self.current_token.type}"
            )

    # Grammar: Expr → Term ((+|-) Term)*
    def parse_expr(self):
        node = self.parse_term()
        while self.current_token.type in ('PLUS', 'MINUS'):
            op = self.current_token
            self.eat(op.type)
            right = self.parse_term()
            node = (op.type, node, right)
        return node

    # Grammar: Term → Factor ((*|/) Factor)*
    def parse_term(self):
        node = self.parse_factor()
        while self.current_token.type in ('MUL', 'DIV'):
            op = self.current_token
            self.eat(op.type)
            right = self.parse_factor()
            node = (op.type, node, right)
        return node

    # Grammar: Factor → INT | ID | '(' Expr ')'
    def parse_factor(self):
        token = self.current_token
        if token.type in ('INT', 'ID'):
            self.eat(token.type)
            return (token.type, token.value)
        elif token.type == 'LPAREN':
            self.eat('LPAREN')
            node = self.parse_expr()
            self.eat('RPAREN')
            return node
        elif token.type == 'MINUS':  # handle unary minus
            self.eat('MINUS')
            node = self.parse_factor()
            return ('NEG', node)
        else:
            raise SyntaxError(
                f"Unexpected token {token.type} at position {token.position}"
            )

    def parse(self):
        result = self.parse_expr()
        if self.current_token.type != 'EOF':
            raise SyntaxError(f"Unexpected token after valid expression: {self.current_token}")
        return result


In [8]:
tokens = tokenize("2 + 3 * 4")
parser = Parser(tokens)
ast = parser.parse()
print(ast)


('PLUS', ('INT', '2'), ('MUL', ('INT', '3'), ('INT', '4')))


In [10]:
import re

# =======================================
# TOKENIZER
# =======================================

TOKEN_SPEC = [
    ('INT',   r'\d+'),
    ('ID',    r'[A-Za-z_]\w*'),
    ('PLUS',  r'\+'),
    ('MINUS', r'-'),
    ('MUL',   r'\*'),
    ('DIV',   r'/'),
    ('LPAREN', r'\('),
    ('RPAREN', r'\)'),
    ('SKIP',  r'[ \t]+'),
    ('MISMATCH', r'.'),
]

class Token:
    def __init__(self, type_, value, position):
        self.type = type_
        self.value = value
        self.position = position
    def __repr__(self):
        if self.value is not None:
            return f"{self.type}({self.value})"
        return self.type

def tokenize(text):
    tokens = []
    pos = 0
    pattern = '|'.join(f'(?P<{name}>{regex})' for name, regex in TOKEN_SPEC)
    for m in re.finditer(pattern, text):
        kind = m.lastgroup
        value = m.group()
        if kind == 'SKIP':
            continue
        elif kind == 'MISMATCH':
            raise SyntaxError(f"Invalid character '{value}' at position {m.start()}")
        tokens.append(Token(kind, value, m.start()))
    tokens.append(Token('EOF', None, len(text)))
    return tokens

# =======================================
# PARSER
# =======================================

class Parser:
    def __init__(self, tokens):
        self.tokens = tokens
        self.pos = 0
        self.current_token = self.tokens[self.pos]

    def error(self, message):
        raise SyntaxError(f"Syntax Error at position {self.current_token.position}: {message}")

    def eat(self, token_type):
        if self.current_token.type == token_type:
            self.pos += 1
            self.current_token = self.tokens[self.pos]
        else:
            self.error(f"expected {token_type}, found {self.current_token.type}")

    def parse_expr(self):
        node = self.parse_term()
        while self.current_token.type in ('PLUS', 'MINUS'):
            op = self.current_token.type
            self.eat(op)
            right = self.parse_term()
            node = (op, node, right)
        return node

    def parse_term(self):
        node = self.parse_factor()
        while self.current_token.type in ('MUL', 'DIV'):
            op = self.current_token.type
            self.eat(op)
            right = self.parse_factor()
            node = (op, node, right)
        return node

    def parse_factor(self):
        token = self.current_token
        if token.type in ('INT', 'ID'):
            self.eat(token.type)
            return (token.type, token.value)
        elif token.type == 'LPAREN':
            self.eat('LPAREN')
            node = self.parse_expr()
            self.eat('RPAREN')
            return node
        elif token.type == 'MINUS':  # unary minus
            self.eat('MINUS')
            node = self.parse_factor()
            return ('NEG', node)
        else:
            self.error("expected INT/ID/LPAREN")

    def parse(self):
        node = self.parse_expr()
        if self.current_token.type != 'EOF':
            self.error("extra input after valid expression")
        return node

# =======================================
# EVALUATOR
# =======================================

def evaluate(node, env=None):
    if env is None:
        env = {}

    nodetype = node[0]

    if nodetype == 'INT':
        return int(node[1])
    elif nodetype == 'ID':
        name = node[1]
        if name in env:
            return env[name]
        else:
            raise NameError(f"Undefined variable '{name}'")
    elif nodetype == 'PLUS':
        return evaluate(node[1], env) + evaluate(node[2], env)
    elif nodetype == 'MINUS':
        return evaluate(node[1], env) - evaluate(node[2], env)
    elif nodetype == 'MUL':
        return evaluate(node[1], env) * evaluate(node[2], env)
    elif nodetype == 'DIV':
        return evaluate(node[1], env) / evaluate(node[2], env)
    elif nodetype == 'NEG':
        return -evaluate(node[1], env)
    else:
        raise ValueError(f"Unknown node type: {nodetype}")

def pretty_print(node, indent=0):
    """Nicely print the AST tree structure."""
    prefix = "  " * indent
    nodetype = node[0]

    if nodetype in ('INT', 'ID'):
        print(f"{prefix}{nodetype}: {node[1]}")
    elif nodetype == 'NEG':
        print(f"{prefix}NEG")
        pretty_print(node[1], indent + 1)
    elif nodetype in ('PLUS', 'MINUS', 'MUL', 'DIV'):
        print(f"{prefix}{nodetype}")
        pretty_print(node[1], indent + 1)
        pretty_print(node[2], indent + 1)
    else:
        print(f"{prefix}{nodetype}?")  # catch unexpected nodes


# =======================================
# DRIVER / TEST
# =======================================
def run_expression(expr, env=None):
    print(f"Expression: {expr}")
    tokens = tokenize(expr)
    parser = Parser(tokens)
    ast = parser.parse()
    print("AST structure:")
    pretty_print(ast)
    result = evaluate(ast, env)
    print("Result:", result)
    print("-" * 40)


# =======================================
# EXAMPLES
# =======================================

if __name__ == "__main__":
    run_expression("2 + 3 * 4")
    run_expression("(2 + 3) * 4")
    run_expression("-5 + 10 / 2")
    run_expression("x * 2 + 1", env={"x": 7})


Expression: 2 + 3 * 4
AST structure:
PLUS
  INT: 2
  MUL
    INT: 3
    INT: 4
Result: 14
----------------------------------------
Expression: (2 + 3) * 4
AST structure:
MUL
  PLUS
    INT: 2
    INT: 3
  INT: 4
Result: 20
----------------------------------------
Expression: -5 + 10 / 2
AST structure:
PLUS
  NEG
    INT: 5
  DIV
    INT: 10
    INT: 2
Result: 0.0
----------------------------------------
Expression: x * 2 + 1
AST structure:
PLUS
  MUL
    ID: x
    INT: 2
  INT: 1
Result: 15
----------------------------------------


In [11]:
from collections import defaultdict

# Grammar definition
grammar = {
    'E':  [['T', "E'"]],
    "E'": [['+', 'T', "E'"], ['ε']],
    'T':  [['F', "T'"]],
    "T'": [['*', 'F', "T'"], ['ε']],
    'F':  [['(', 'E', ')'], ['id']]
}

terminals = {'+', '*', '(', ')', 'id'}
nonterminals = set(grammar.keys())

# ===========================
# FIRST SETS
# ===========================

FIRST = defaultdict(set)

def first(symbol):
    if symbol in terminals:
        return {symbol}
    elif symbol == 'ε':
        return {'ε'}
    if FIRST[symbol]:
        return FIRST[symbol]

    for production in grammar[symbol]:
        for sym in production:
            sym_first = first(sym)
            FIRST[symbol].update(sym_first - {'ε'})
            if 'ε' not in sym_first:
                break
        else:
            FIRST[symbol].add('ε')
    return FIRST[symbol]

for nt in nonterminals:
    first(nt)

print("FIRST sets:")
for nt in nonterminals:
    print(f"{nt}: {FIRST[nt]}")

# ===========================
# FOLLOW SETS
# ===========================

FOLLOW = defaultdict(set)
FOLLOW['E'].add('$')  # $ is end-of-input symbol

def follow(symbol):
    for nt in grammar:
        for production in grammar[nt]:
            for i, sym in enumerate(production):
                if sym == symbol:
                    rest = production[i+1:]
                    if rest:
                        first_rest = set()
                        for r in rest:
                            first_r = first(r)
                            first_rest.update(first_r - {'ε'})
                            if 'ε' in first_r:
                                continue
                            else:
                                break
                        else:
                            FOLLOW[sym].update(FOLLOW[nt])
                        FOLLOW[sym].update(first_rest)
                    else:
                        FOLLOW[sym].update(FOLLOW[nt])

for _ in range(10):  # iterate to convergence
    for nt in nonterminals:
        follow(nt)

print("\nFOLLOW sets:")
for nt in nonterminals:
    print(f"{nt}: {FOLLOW[nt]}")

# ===========================
# LL(1) PARSE TABLE
# ===========================

parse_table = defaultdict(dict)

for nt in grammar:
    for production in grammar[nt]:
        first_prod = set()
        for sym in production:
            sym_first = first(sym)
            first_prod.update(sym_first - {'ε'})
            if 'ε' not in sym_first:
                break
        else:
            first_prod.add('ε')

        for terminal in first_prod - {'ε'}:
            parse_table[nt][terminal] = production

        if 'ε' in first_prod:
            for terminal in FOLLOW[nt]:
                parse_table[nt][terminal] = production

print("\nLL(1) Parse Table:")
for nt in parse_table:
    for t in parse_table[nt]:
        print(f"M[{nt}, {t}] = {parse_table[nt][t]}")


FIRST sets:
E: {'id', '('}
T': {'*', 'ε'}
F: {'id', '('}
E': {'ε', '+'}
T: {'id', '('}

FOLLOW sets:
E: {')', '$'}
T': {')', '$', '+'}
F: {')', '*', '$', '+'}
E': {')', '$'}
T: {')', '$', '+'}

LL(1) Parse Table:
M[E, id] = ['T', "E'"]
M[E, (] = ['T', "E'"]
M[E', +] = ['+', 'T', "E'"]
M[E', )] = ['ε']
M[E', $] = ['ε']
M[T, id] = ['F', "T'"]
M[T, (] = ['F', "T'"]
M[T', *] = ['*', 'F', "T'"]
M[T', )] = ['ε']
M[T', $] = ['ε']
M[T', +] = ['ε']
M[F, (] = ['(', 'E', ')']
M[F, id] = ['id']


In [13]:
class LL1Parser:
    def __init__(self, grammar, parse_table, start_symbol):
        self.grammar = grammar
        self.parse_table = parse_table
        self.start_symbol = start_symbol

    def parse(self, tokens):
        tokens.append('$')  # end-of-input marker
        stack = ['$']
        stack.append(self.start_symbol)
        input_pos = 0
        derivation = []
        parse_tree = {self.start_symbol: []}  # simple tree using dicts

        # helper to build tree nodes
        node_stack = [parse_tree]

        while stack:
            top = stack.pop()
            current_token = tokens[input_pos]

            if top in terminals or top == '$':
                if top == current_token:
                    input_pos += 1
                else:
                    raise SyntaxError(f"Unexpected token {current_token}, expected {top}")
            else:  # nonterminal
                if current_token in self.parse_table[top]:
                    production = self.parse_table[top][current_token]
                    derivation.append((top, production))
                    # update stack: push in reverse
                    for sym in reversed(production):
                        if sym != 'ε':
                            stack.append(sym)
                    # update parse tree
                    node = node_stack.pop()
                    node[top] = []
                    for sym in production:
                        if sym != 'ε':
                            node[top].append({sym: []})
                            node_stack.append(node[top][-1])
                else:
                    raise SyntaxError(f"No rule for {top} with lookahead {current_token}")

        return derivation, parse_tree

def pretty_print_tree(node, indent=0):
    prefix = "  " * indent
    for key, children in node.items():
        print(f"{prefix}{key}")
        for child in children:
            pretty_print_tree(child, indent + 1)

# ===========================
# TEST LL(1) PARSER
# ===========================

tokens = ['id', '+', 'id', '*', 'id']
parser_driver = LL1Parser(grammar, parse_table, 'E')
derivation_steps, tree = parser_driver.parse(tokens)

print("Leftmost Derivation Steps:")
for step in derivation_steps:
    print(step)


print("\nParse Tree:")
pretty_print_tree(tree)



Leftmost Derivation Steps:
('E', ['T', "E'"])
('T', ['F', "T'"])
('F', ['id'])
("T'", ['ε'])
("E'", ['+', 'T', "E'"])
('T', ['F', "T'"])
('F', ['id'])
("T'", ['*', 'F', "T'"])
('F', ['id'])
("T'", ['ε'])
("E'", ['ε'])

Parse Tree:
E
  T
  E'
  T
    F
    E'
      +
      T
      E'
      T
        F
        T'
        F
          id
          T'
            *
            F
            E'
            T'
            F
              id
              T'
    T'
    F
      id
      T'
